# Ingesta: del dataset de Kaggle al Volume de Bronze
Descarga desde Kaggle los 7 CSV que usa el proyecto y los deja en el Volume `dunnhumby.bronze.raw_files`, de donde los lee `01_bronze`.

- **Fuente:** [Dunnhumby - The Complete Journey](https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey) (`frtgnn/dunnhumby-the-complete-journey`). Es un dataset público, así que la descarga no necesita token.
- **`causal_data.csv` no se descarga:** es el archivo más pesado del dataset y ninguna pregunta de negocio lo usa.
- **Plan B:** si Kaggle no responde y el archivo ya está en el Volume (por ejemplo, porque se subió a mano), se usa esa copia y el proceso continúa.


In [0]:
# SECCIÓN 0 · Librería de Kaggle (en serverless se instala en cada corrida; %pip reinicia Python, por eso va primero)
%pip install "kagglehub>=0.3" --quiet


In [0]:
# SECCIÓN 1 · Descarga archivo por archivo y copia al Volume
import os, shutil
import kagglehub

dbutils.widgets.text("dataset_kaggle", "frtgnn/dunnhumby-the-complete-journey")
DATASET = dbutils.widgets.get("dataset_kaggle")
DESTINO = "/Volumes/dunnhumby/bronze/raw_files"

# Los mismos 7 archivos que carga 01_bronze
ARCHIVOS = ["campaign_desc", "campaign_table", "coupon", "coupon_redempt",
            "hh_demographic", "product", "transaction_data"]

for nombre in ARCHIVOS:
    destino = f"{DESTINO}/{nombre}.csv"
    try:
        # path= baja solo ese archivo, no el zip completo del dataset
        origen = kagglehub.dataset_download(DATASET, path=f"{nombre}.csv")
        shutil.copyfile(origen, destino)   # el Volume se ve como una carpeta normal, así que se copia como un archivo local
        print(f"{nombre}.csv: descargado de Kaggle ({os.path.getsize(destino):,} bytes)")
    except Exception as error:
        # Plan B: si ya existe una copia en el Volume se usa esa; si no, el Workflow se detiene aquí
        if not os.path.exists(destino):
            raise RuntimeError(f"No se pudo descargar {nombre}.csv y no hay copia en el Volume") from error
        print(f"{nombre}.csv: Kaggle no respondió, se usa la copia existente del Volume ({error.__class__.__name__})")


In [0]:
# SECCIÓN 2 · Verificación: los 7 CSV deben estar en el Volume y no estar vacíos
# Esperado: 7 archivos; transaction_data.csv es el más grande (unos 140 MB)
en_volume = {f.name: f.size for f in dbutils.fs.ls(DESTINO)}
faltantes = [f"{n}.csv" for n in ARCHIVOS if en_volume.get(f"{n}.csv", 0) == 0]
assert not faltantes, f"Faltan o están vacíos en el Volume: {faltantes}"

display(spark.createDataFrame(
    [(f"{n}.csv", en_volume[f"{n}.csv"]) for n in ARCHIVOS], ["archivo", "bytes"]
))
